# 12 · Klassen und Objekte

**Lernziele**

- Du definierst Klassen mit Attributen und Methoden.
- Du erzeugst Objekte und arbeitest mit ihnen.
- Du gibst Objekte lesbar aus und vergleichst sie.
- Du nutzt `@dataclass` für reine Datenklassen.
- Du legst feste Auswahlwerte als `Enum` an.
- Du erweiterst Klassen durch Vererbung und überschreibst Methoden.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 4 und 6.

## Warum Klassen?

Bisher hast du Daten (z. B. Dictionaries) und Funktionen getrennt. Eine **Klasse** bündelt beides: Sie beschreibt, welche Daten ein Ding hat (**Attribute**) und was es kann (**Methoden**). Ein **Objekt** ist ein konkretes Exemplar einer Klasse.

| Klasse (Bauplan) | Objekte (Exemplare) |
| --- | --- |
| `Car` | Toms Audi, Jims VW |
| `BankAccount` | Konto 4711, Konto 0815 |

## Eine erste Klasse

`__init__` wird beim Erzeugen aufgerufen und legt die Attribute an. `self` ist das Objekt selbst.

In [ ]:
class Car:
    def __init__(self, brand, color, horse_power):
        self.brand = brand
        self.color = color
        self.horse_power = horse_power

    def paint(self, new_color):
        self.color = new_color

    def apply_tuning_kit(self):
        self.horse_power += 150


my_car = Car("VW", "orange", 123)
print(my_car.brand, my_car.color, my_car.horse_power)

my_car.paint("green")
my_car.apply_tuning_kit()
print(my_car.color, my_car.horse_power)

Eine Klasse, mehrere Objekte, und `self` zeigt auf das Objekt, für das die Methode läuft:

![Die Klasse Car als Bauplan erzeugt zwei Objekte mit eigenen Attributwerten; my_car und self zeigen auf dasselbe Objekt, dessen Farbe paint von orange auf green ändert.](../assets/12_klasse_objekte.png)

**Kurzübung 1 (self vergessen):** `Contact("Ada Lovelace", "ada@example.com").signature()` soll `Ada Lovelace <ada@example.com>` liefern. Im Code stecken zwei Fehler. Finde und behebe beide.

In [ ]:
class Contact:
    def __init__(self, name: str, email: str) -> None:
        name = name
        email = email

    def signature() -> str:
        return f"{self.name} <{self.email}>"


ada = Contact("Ada Lovelace", "ada@example.com")
print(ada.signature())

In [ ]:
# Deine Lösung


## Lesbare Ausgabe mit `__str__`

Ohne Hilfe sieht ein Objekt in der Ausgabe kryptisch aus:

In [ ]:
print(my_car)

Methoden mit doppelten Unterstrichen (**Dunder-Methoden**) haben eine besondere Bedeutung. `__str__` legt fest, was `print()` zeigt:

In [ ]:
class Car:
    def __init__(self, brand, color, horse_power):
        self.brand = brand
        self.color = color
        self.horse_power = horse_power

    def __str__(self):
        return f"{self.brand} ({self.color}, {self.horse_power} PS)"


print(Car("Audi", "blue", 275))

**Kurzübung 2 (Rechnung lesbar ausgeben):** Ergänze in `Invoice` eine Methode `__str__`, sodass `print(invoice)` den Text `Rechnung 2026-117: 249.90 €` zeigt.

- **Untersuchen:** Was passiert, wenn `__str__` den Text mit `print()` ausgibt, statt ihn mit `return` zurückzugeben?

In [ ]:
class Invoice:
    def __init__(self, number: str, amount: float) -> None:
        self.number = number
        self.amount = amount


invoice = Invoice("2026-117", 249.9)
print(invoice)

In [ ]:
# Deine Lösung


## Objekte vergleichen

`==` vergleicht standardmäßig, ob es **dasselbe** Objekt ist, nicht ob die Inhalte gleich sind:

In [ ]:
toms_car = Car("Audi", "blue", 275)
jims_car = Car("Audi", "blue", 275)
print(toms_car == jims_car)

Sollen gleiche Inhalte als gleich gelten, schreibst du die Dunder-Methode `__eq__`. Sie bekommt das andere Objekt als `other`:

- `isinstance(other, Car)` prüft zuerst, ob `other` überhaupt ein `Car` ist.
- Wenn nicht, gibt `__eq__` den Sonderwert `NotImplemented` zurück. Python fragt dann das andere Objekt; kann auch das nicht vergleichen, ist das Ergebnis `False`.
- Sonst vergleicht `__eq__` die Attribute, hier bequem als Tupel.

Dazu kommt `__repr__`: Es liefert eine Darstellung für Entwickler, am besten so, wie das Objekt im Code entstünde. `print()` nutzt `__str__`, Listen und die Anzeige der letzten Zeile im Notebook nutzen `__repr__` (fehlt `__str__`, nimmt auch `print()` das `__repr__`). `!r` im f-String setzt einen Wert so ein, wie er im Code stünde, Strings also mit Anführungszeichen.

In [ ]:
class Car:
    def __init__(self, brand, color, horse_power):
        self.brand = brand
        self.color = color
        self.horse_power = horse_power

    def __eq__(self, other):
        if not isinstance(other, Car):
            return NotImplemented
        return (self.brand, self.color, self.horse_power) == (
            other.brand,
            other.color,
            other.horse_power,
        )

    def __repr__(self):
        return f"Car({self.brand!r}, {self.color!r}, {self.horse_power})"


toms_car = Car("Audi", "blue", 275)
jims_car = Car("Audi", "blue", 275)
print(toms_car == jims_car)  # True: gleiche Inhalte
print(toms_car is jims_car)  # False: trotzdem zwei Objekte
print(toms_car == "Audi")  # False: kein Car
print([toms_car, Car("VW", "green", 273)])  # Listen nutzen __repr__

**Kurzübung 3 (Zwei gleiche Kontakte):** Welche Ausgabe erzeugt `Contact("Ada", "ada@example.com") == Contact("Ada", "ada@example.com")`, und warum? Ergänze `__eq__` wie bei `Car`, sodass der Vergleich `True` liefert.

- **Untersuchen:** Was liefert `ada is Contact("Ada", "ada@example.com")`, nachdem `__eq__` da ist?
- **Ändern (optional):** Ergänze `__repr__`, sodass `print([ada])` den Text `[Contact('Ada', 'ada@example.com')]` zeigt, und vergleiche E-Mail-Adressen ohne Groß-/Kleinschreibung (`casefold()`, Notebook 05).

In [ ]:
class Contact:
    def __init__(self, name: str, email: str) -> None:
        self.name = name
        self.email = email


ada = Contact("Ada", "ada@example.com")
print(Contact("Ada", "ada@example.com") == Contact("Ada", "ada@example.com"))
print([ada])

In [ ]:
# Deine Lösung


`__eq__` und `__repr__` selbst zu schreiben ist Fleißarbeit. Einfacher geht es mit einer **Datenklasse**.

## Datenklassen mit `@dataclass`

Für Klassen, die vor allem Daten halten, erzeugt `@dataclass` automatisch `__init__`, `__repr__` und `__eq__`. Die Attribute werden mit Type Hints angegeben.

Eine Zeile mit `@` direkt über einer Klasse oder Funktion heißt **Dekorator**: Sie erweitert das, was darunter steht, hier um die erzeugten Methoden. `@property` weiter unten ist ebenfalls ein Dekorator.

In [ ]:
from dataclasses import dataclass


@dataclass
class Car:
    brand: str
    color: str
    horse_power: int

    def apply_tuning_kit(self) -> None:
        self.horse_power += 150


toms_car = Car("Audi", "blue", 275)
jims_car = Car("Audi", "blue", 275)
print(toms_car)
print(toms_car == jims_car)

> Faustregel: Für reine Daten `@dataclass`, für Objekte mit viel Verhalten eine normale Klasse. Beides sind ganz normale Klassen.

### Klassenattribute

Bisher hingen alle Attribute mit `self.` an einem einzelnen Objekt (**Instanzattribute**). Steht eine Zuweisung direkt im Klassenkörper, ohne `self.`, ist sie ein **Klassenattribut**: Es gehört der Klasse, und alle Objekte teilen sich denselben Wert. Das passt für Konstanten wie einen Höchstwert; Großbuchstaben zeigen wie bei Konstanten sonst, dass sich der Wert nicht ändern soll.

In [ ]:
class Parcel:
    MAX_WEIGHT = 31.5  # Klassenattribut: gilt für alle Pakete

    def __init__(self, weight: float) -> None:
        self.weight = weight  # Instanzattribut: je Paket verschieden

    def is_allowed(self) -> bool:
        return self.weight <= self.MAX_WEIGHT  # über self oder Parcel lesbar


small, heavy = Parcel(2.0), Parcel(40.0)
print(small.MAX_WEIGHT, heavy.MAX_WEIGHT, Parcel.MAX_WEIGHT)  # derselbe Wert
print(small.is_allowed(), heavy.is_allowed())

Was passiert, wenn ein Klassenattribut veränderlich ist, etwa eine Liste, zeigt die nächste Kurzübung.

**Kurzübung 4 (Alle Teams teilen eine Liste):** Ada kommt ins Team Vertrieb, steht danach aber auch im Team Support. Finde den Fehler und behebe ihn mit einer Datenklasse. Was passiert, wenn du dort `members: list[str] = []` schreibst? Die Fehlermeldung nennt den Ausweg: `field(default_factory=list)` aus dem Modul `dataclasses`.

In [ ]:
class Team:
    members = []

    def __init__(self, name: str) -> None:
        self.name = name


sales, support = Team("Vertrieb"), Team("Support")
sales.members.append("Ada")
print(support.members)

In [ ]:
# Deine Lösung


## Interne Attribute

Python kennt kein `private`. Die **Konvention**: Ein Unterstrich am Anfang bedeutet „intern, bitte nicht von außen ändern“. Der Zugriff erfolgt über Methoden.

In [ ]:
class ClickCounter:
    def __init__(self):
        self._value = 0

    @property
    def value(self) -> int:
        return self._value

    def increment(self) -> None:
        self._value += 1

    def reset(self) -> None:
        self._value = 0


counter = ClickCounter()
counter.increment()
counter.increment()
print(counter.value)
counter.reset()
print(counter.value)

`@property` macht `value` von außen **lesbar wie ein Attribut**, aber nicht schreibbar:

In [ ]:
counter.value = 100

**Kurzübung 5 (Wartenummern im Bürgerbüro):** Schreibe eine Klasse `TicketMachine`, die die zuletzt vergebene Nummer intern in `_last_number` speichert (Start: 0). Die Methode `draw()` erhöht sie um 1 und gibt die neue Nummer zurück, die Property `last_number` macht sie von außen lesbar. Ziehe drei Nummern und gib danach `last_number` aus. Was passiert bei `machine.last_number = 0`?

In [ ]:
# Deine Lösung


## Aufzählungen mit `Enum`

Manche Werte haben nur wenige erlaubte Möglichkeiten: Ampelfarben, Spielkartenfarben, der Status einer Bestellung. Als Strings wie `"rot"` sind sie anfällig für Tippfehler: `"Rot"` oder `"rto"` fällt erst spät oder gar nicht auf. Eine **Aufzählung** (*Enumeration*) legt die erlaubten Werte als Klasse fest. Die Mitglieder schreibt man in Großbuchstaben, weil es Konstanten sind:

In [ ]:
from enum import Enum, StrEnum, auto


class TrafficLight(Enum):
    RED = 1
    YELLOW = 2
    GREEN = 3


light = TrafficLight.RED
print(light)
print(light.name, light.value)
print(light == TrafficLight.RED)

Jedes Mitglied hat einen Namen (`name`) und einen Wert (`value`). Verglichen wird mit `==` oder `is`. Ein Tippfehler fällt sofort auf, statt still falsch zu laufen:

In [ ]:
TrafficLight.RDE

Über eine Enum kannst du iterieren, in der Reihenfolge der Definition:

In [ ]:
for color in TrafficLight:
    print(color.name, color.value)

print(TrafficLight(2))  # aus einem gespeicherten Wert wird wieder ein Mitglied

Sind dir die konkreten Werte egal, vergibt `auto()` sie. `StrEnum` verhält sich zusätzlich wie ein String und bekommt mit `auto()` den Namen in Kleinbuchstaben als Wert. Das ist praktisch, wenn die Werte in Dateien, JSON oder Ausgaben landen:

In [ ]:
class OrderStatus(StrEnum):
    OPEN = auto()  # Wert: "open"
    SHIPPED = auto()
    DELIVERED = auto()


status = OrderStatus.SHIPPED
print(status, status == "shipped")
print(f"Status: {status}")

Enums und `match` (Exkurs in Notebook 03) passen gut zusammen. Weil `TrafficLight.RED` einen Punkt enthält, **vergleicht** `case` mit diesem Wert, statt ihn einzufangen:

In [ ]:
def next_light(light: TrafficLight) -> TrafficLight:
    match light:  # vereinfacht, ohne Rot-Gelb
        case TrafficLight.RED:
            return TrafficLight.GREEN
        case TrafficLight.GREEN:
            return TrafficLight.YELLOW
        case TrafficLight.YELLOW:
            return TrafficLight.RED
        case _:
            raise ValueError(f"Unbekannte Ampelphase: {light}")


light = TrafficLight.RED
for _ in range(4):
    print(light.name)
    light = next_light(light)

**Kurzübung 6 (Kleidergrößen als Enum):** Ein Shop sortiert Größen als Text: `sorted(["XL", "S", "L", "M"])`. Welches Ergebnis entsteht? Schreibe eine Enum `Size` mit den Werten `S = 1`, `M = 2`, `L = 3` und `XL = 4` und sortiere `[Size.XL, Size.S, Size.L, Size.M]` mit `sorted(..., key=lambda size: size.value)`. Erwartet: S, M, L, XL.

- **Untersuchen:** Was passiert beim Tippfehler `Size.XLL`, und was, wenn der Text `"XLL"` in der Liste oben steht?
- **Ändern:** Ergänze `XXL` so, dass sie hinter `XL` einsortiert wird.

In [ ]:
print(sorted(["XL", "S", "L", "M"]))

In [ ]:
# Deine Lösung


In [ ]:
# Deine Lösung


## Vererbung

Eine Klasse kann von einer anderen **erben**: Sie übernimmt alles und kann Methoden ergänzen oder **überschreiben**. Mit `super()` rufst du die Version der Elternklasse auf.

Beispiel: Ein Punktezähler, der bei 100 überläuft und dafür ein Bonusleben vergibt.

In [ ]:
class CounterWithOverflow(ClickCounter):
    MAX_VALUE = 100

    def __init__(self):
        super().__init__()
        self.overflow_count = 0

    def increment(self) -> None:
        super().increment()
        if self.value >= self.MAX_VALUE:
            self._value = 0
            self.overflow_count += 1


points = CounterWithOverflow()
for _ in range(2026):
    points.increment()

print(f"Punkte: {points.value}, Bonusleben: {points.overflow_count}")
print(isinstance(points, ClickCounter))  # ein CounterWithOverflow ist auch ein ClickCounter

Was `CounterWithOverflow` überschreibt, unverändert erbt und neu mitbringt:

![CounterWithOverflow erbt von ClickCounter: __init__ und increment sind überschrieben und rufen die Elternversion mit super() auf, value und reset sind unverändert geerbt, MAX_VALUE und overflow_count sind neu.](../assets/12_vererbung.png)

Unterschiedliche Klassen können dieselbe Methode unterschiedlich umsetzen. Der aufrufende Code muss den genauen Typ nicht kennen (**Polymorphie**):

In [ ]:
import math


class Shape:
    def area(self) -> float:
        raise NotImplementedError


class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return math.pi * self.radius ** 2


for shape in [Rectangle(3, 4), Circle(1)]:
    print(f"{type(shape).__name__}: {shape.area():.2f}")

> **Vererbung sparsam einsetzen.** Sie passt, wenn gilt „X **ist ein** Y“. Oft ist es einfacher, ein Objekt als Attribut in einem anderen zu verwenden („X **hat ein** Y“).

**Kurzübung 7 (Besprechungsraum voll):** Leite von `ClickCounter` eine Klasse `RoomCounter` ab, die Personen in einem Besprechungsraum mit 8 Plätzen zählt. Lege die Grenze als Klassenattribut `MAX_VALUE = 8` an und überschreibe `increment()`: Ist der Raum voll, gibt die Methode `Raum voll` aus, sonst zählt sie mit `super().increment()` weiter. Rufe `increment()` zehnmal auf. Welche Ausgabe erwartest du, und welchen Wert hat `value` danach?

In [ ]:
# Deine Lösung


## Zusammenfassung

- `class Name:` mit `__init__(self, ...)` legt Attribute an, Methoden bekommen `self`.
- `__str__` steuert die Ausgabe mit `print()`, `__repr__` die Darstellung in Listen und im Notebook.
- `__eq__` legt fest, wann `==` zwei Objekte für gleich hält; ohne `__eq__` nur bei demselben Objekt.
- Klassenattribute stehen ohne `self.` im Klassenkörper und gelten für alle Objekte.
- `@dataclass` (ein Dekorator) erzeugt `__init__`, `__repr__` und `__eq__` automatisch.
- `_name` heißt „intern“; `@property` macht Werte lesbar, aber nicht schreibbar.
- `Enum` für feste Auswahlwerte, `StrEnum` wenn sie sich wie Strings verhalten sollen; passt zu `match`.
- Vererbung: `class Child(Parent):`, `super().methode()` ruft die Elternversion auf.